<a id='set-comp'></a>

## 7. 🧩 Pattern 7: Set Comprehensions — {expr for x in iterable} — LC 128, 217, 242, 349, 383, 567, 705

---

```
PROBLEM:
  LC 128 — Longest Consecutive: unique vals in O(1) lookup set
  LC 217 — Contains Duplicate: seen set — hit means duplicate
  LC 242 — Valid Anagram: set(s) == set(t) + freq check
  LC 349 — Intersection of Two Arrays: set(a) & set(b)
  LC 383 — Ransom Note: available chars as set or freq map
  LC 567 — Permutation in String: sliding window + set/freq
  LC 705 — Design HashSet: membership via set

SYNTAX — three core forms:
  {expr for x in iterable}           ← basic set comp
  {expr for x in iterable if cond}   ← filtered set comp
  set(iterable)                      ← constructor shorthand (no transform needed)

SET COMP vs set() CONSTRUCTOR:
  set(s)                    ← just deduplicate, no transform — use constructor
  {c.lower() for c in s}   ← transform while deduplicating — use set comp
  {c for c in s if c.isalpha()}  ← filter while deduplicating — use set comp
  Rule: if you're just copying, use set(); if transforming or filtering, use {}.

DEDUPLICATION PATTERN:
  words = ["apple", "fig", "apple", "cherry", "fig"]
  unique = {w for w in words}     ← same as set(words) here, but composable
  unique_upper = {w.upper() for w in words}  ← transform + dedup in one line

MEMBERSHIP TESTING O(1):
  seen = set()
  for num in nums:
      if num in seen:       ← O(1) hash lookup, not O(n) list scan
          return True
      seen.add(num)

DIFFERENCE vs LIST COMP:
  [expr for x in data]   → list  — ordered, duplicates kept, O(n) membership
  {expr for x in data}   → set   — unordered, deduped, O(1) membership
  Choose set comp when: uniqueness matters OR you need fast 'in' checks.

SLOW MOTION TRACE — unique chars from "banana":
  s = "banana"
  unique = {c for c in s}
  step 1: c="b" → {"b"}
  step 2: c="a" → {"b","a"}
  step 3: c="n" → {"b","a","n"}
  step 4: c="a" → already in set, ignored
  step 5: c="n" → already in set, ignored
  step 6: c="a" → already in set, ignored
  result: {"b","a","n"}  — order NOT guaranteed

KEY INSIGHT:
  Set comp is the deduplication + O(1) lookup builder.
  The moment you write 'if x in list', replace that list with a set.

TIME / SPACE:
  Time:  O(n) — one pass to build
  Space: O(k) — k = number of unique elements (k ≤ n)
```

In [ ]:
# Pattern 7: Set Comprehensions
# The deduplication + O(1) lookup builder — replace lists with sets on 'in' checks.

# 1. basic set comp vs constructor
s = "banana"
via_comp   = {c for c in s}          # set comp — dedup while iterating
via_ctor   = set(s)                  # constructor — same result, less composable
via_filter = {c for c in s if c != "a"}   # filtered — can't do this with set()
print(f"via comp   : {via_comp}")
print(f"via ctor   : {via_ctor}")
print(f"filtered   : {via_filter}")

# 2. transform + dedup in one line
words = ["apple", "fig", "apple", "cherry", "fig"]
unique_upper = {w.upper() for w in words}   # dedup + uppercase in one pass
print(f"unique upper: {unique_upper}")

# 3. set operations — intersection / difference (LC 349 pattern)
a = [1, 2, 2, 3, 4]
b = [2, 4, 4, 6]
intersection = set(a) & set(b)    # O(min(m,n)) — elements in both
difference   = set(a) - set(b)    # O(m) — elements only in a
union        = set(a) | set(b)    # O(m+n) — all unique elements
print(f"intersection: {intersection}")
print(f"difference  : {difference}")
print(f"union       : {union}")


def contains_duplicate_set(nums: list) -> bool:
    """
    LC 217 — Contains Duplicate
    Approach: seen set — first repeated element triggers immediate return.
    Args:
        nums (list[int]): integer array.
    Returns:
        bool: True if any value appears at least twice.
    Time:  O(n) — one pass, early exit on first duplicate
    Space: O(n) — seen set stores at most n elements
    """
    seen = set()

    # slow motion on nums = [1, 2, 3, 1]:
    # num=1: 1 not in {} → seen={1}
    # num=2: 2 not in {1} → seen={1,2}
    # num=3: 3 not in {1,2} → seen={1,2,3}
    # num=1: 1 in {1,2,3} → True  ← early exit here

    for num in nums:
        if num in seen:       # O(1) hash lookup — not O(n) list scan
            return True
        seen.add(num)         # record for future lookups
    return False


def intersection_of_two_arrays(nums1: list, nums2: list) -> list:
    """
    LC 349 — Intersection of Two Arrays
    Approach: set intersection — & operator handles dedup and membership in one shot.
    Args:
        nums1 (list[int]): first array.
        nums2 (list[int]): second array.
    Returns:
        list[int]: unique elements present in both arrays.
    Time:  O(m + n) — build two sets, intersect
    Space: O(m + n) — two sets
    """
    set1 = {x for x in nums1}   # set comp — explicit; same as set(nums1)
    set2 = {x for x in nums2}

    # slow motion on nums1=[1,2,2,1], nums2=[2,2]:
    # set1 = {1, 2}
    # set2 = {2}
    # set1 & set2 = {2}  ← only elements in BOTH

    return list(set1 & set2)    # & = intersection, result is a set → convert to list


def test_harness(fn):
    tests = [
        ([1, 2, 3, 1],  True),             # duplicate: 1
        ([1, 2, 3, 4],  False),            # no duplicate
        ([1, 1],        True),             # immediate duplicate
        ([1],           False),            # single element
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(contains_duplicate_set)

# intersection demo
print(intersection_of_two_arrays([1, 2, 2, 1], [2, 2]))    # [2]
print(intersection_of_two_arrays([4, 9, 5], [9, 4, 9, 8, 4]))  # [9, 4]

print("set_comprehensions defined.")